# 🛒 Naivas Online Product Scraper

This notebook scrapes **product names and prices** from 4 Naivas Online categories:
- 🥫 Food Cupboard
- 🥦 Fresh Food
- 📱 Electronics
- 🍾 Naivas Liquor

**How to use:** Run each cell one by one from top to bottom using the ▶ button or `Shift + Enter`.

In [13]:
# =============================================================
# CELL 1 — IMPORT LIBRARIES
# =============================================================
# Think of libraries as tools we borrow to make work easier.
#
# requests       : sends a request to a website (like your browser)
# BeautifulSoup  : reads and understands the HTML of a webpage
# pandas         : organises data into a table (like Excel)
# time           : lets us pause between requests (polite to server)
# =============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


In [26]:
# =============================================================
# CELL 2 — SETTINGS & CONFIGURATION
# =============================================================
# All important settings are defined here in one place.
# If anything needs changing, you only edit this cell.
# =============================================================

# --- HEADERS ---
# When a browser visits a website it sends info about itself.
# Without this, websites detect us as a bot and block us.
# We pretend to be a normal Chrome browser using these headers.
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/122.0.0.0 Safari/537.36'
    ),
    # Tell the site we accept normal web pages
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    # Tell the site we prefer English
    'Accept-Language': 'en-US,en;q=0.9',
    # This makes us look like we clicked a link from their homepage
    'Referer': 'https://naivas.online/',
}

# --- DELAY ---
# Seconds to wait between each page request.
# IMPORTANT: Never set this below 2 — too fast = IP ban!
DELAY_SECONDS = 3

# --- CATEGORIES TO SCRAPE ---
# Format: 'Name we give it' : 'URL of the page'
CATEGORIES = {
    'Food Cupboard' : 'https://naivas.online/food-cupboard',
    'Fresh Food'    : 'https://naivas.online/fresh-food',
    'Electronics'   : 'https://naivas.online/electronics',
    'Naivas Liquor' : 'https://naivas.online/naivas-liqour',
}

print('✅ Settings configured!')
print(f'   Will scrape {len(CATEGORIES)} categories:')
for name, url in CATEGORIES.items():
    print(f'   - {name}: {url}')

✅ Settings configured!
   Will scrape 4 categories:
   - Food Cupboard: https://naivas.online/food-cupboard
   - Fresh Food: https://naivas.online/fresh-food
   - Electronics: https://naivas.online/electronics
   - Naivas Liquor: https://naivas.online/naivas-liqour


In [15]:
# =============================================================
# CELL 3 — CREATE A CLOUDSCRAPER SESSION (Replaces requests)
# =============================================================
# cloudscraper handles Cloudflare protection automatically.
# It acts like a real browser so the website won't block us.
# The last line 'session = scraper' means ALL other cells
# stay exactly the same — no other changes needed!
# =============================================================

import cloudscraper

# Create the scraper pretending to be Chrome on Windows
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',   # Pretend to be Chrome
        'platform': 'windows', # Pretend to be on Windows
        'mobile': False        # Pretend to be on desktop
    }
)

# Apply our headers on top of cloudscraper's built-in ones
scraper.headers.update(HEADERS)

# This line means ALL other cells stay unchanged
session = scraper

print('✅ Cloudscraper session created!')
print('   Cloudflare protection will be handled automatically.')

✅ Cloudscraper session created!
   Cloudflare protection will be handled automatically.


In [27]:
# =============================================================
# CELL 4 — FUNCTION: FETCH A PAGE
# =============================================================
# A function is a reusable block of code.
# Instead of writing the same thing over and over, we write
# it once and call it whenever we need it.
#
# This function:
#   1. Takes a URL as input
#   2. Visits that URL like a browser would
#   3. Returns the page HTML
#   4. Handles errors without crashing
# =============================================================

def fetch_page(url):
    """
    Visits a URL and returns its HTML content.
    Returns None if the page could not be loaded.
    """
    try:
        print(f'   🌐 Visiting: {url}')

        # Send a GET request — same as typing a URL in your browser
        # timeout=15 means: give up if no response in 15 seconds
        response = session.get(url, timeout=30)

        # Status code tells us if it worked:
        # 200 = Success ✅  |  403 = Blocked ❌  |  404 = Not Found ❌
        print(f'   📡 Response status: {response.status_code}')

        if response.status_code == 200:
            print('   ✅ Page loaded successfully!')
            return response.text  # Return the raw HTML

        elif response.status_code == 403:
            print('   ❌ Blocked (403) — site is rejecting our request.')
            print('      💡 Try changing headers or increasing delay.')
            return None

        elif response.status_code == 404:
            print('   ❌ Page not found (404) — check the URL is correct.')
            return None

        else:
            print(f'   ⚠️ Unexpected response: {response.status_code}')
            return None

    except requests.exceptions.Timeout:
        # Website took too long to respond
        print('   ❌ Timeout — website took too long. Try again later.')
        return None

    except requests.exceptions.ConnectionError:
        # No internet or wrong URL
        print('   ❌ Connection failed — check your internet connection.')
        return None

    except Exception as e:
        # Any other unexpected error
        print(f'   ❌ Unexpected error: {e}')
        return None


print('✅ fetch_page() function is ready!')

✅ fetch_page() function is ready!


In [7]:
import cloudscraper
print(cloudscraper.__version__)
print("✅ cloudscraper is ready!")

1.2.71
✅ cloudscraper is ready!


In [24]:
# =============================================================
# CELL 5 — EXTRACT PRODUCTS (Updated with correct selectors)
# =============================================================
# Now we know the EXACT HTML structure Naivas uses:
#
# Product card  → <div class="flex flex-col grow">
# Product name  → <span class="line-clamp-2 text-ellipsis">
# Current price → <span class="font-bold text-naivas-green">
# Old price     → <span class="text-red-600 line-through">
# Product link  → <a class="!text-naivas-gray-dark ...">
# =============================================================

def extract_products(html, category_name):
    """
    Reads page HTML and extracts product names and prices.
    Returns a list of products (each product is a dictionary).
    """

    soup = BeautifulSoup(html, 'html.parser')
    products = []

    # -----------------------------------------------------------
    # FIND ALL PRODUCT CARDS
    # -----------------------------------------------------------
    # Each product lives inside a <div class="flex flex-col grow">
    # We find them by looking for the product-price div inside,
    # then navigating to its grandparent (the full card)
    # -----------------------------------------------------------

    price_elements = soup.select('.product-price')
    print(f'   🔍 Found {len(price_elements)} products on page')

    if not price_elements:
        print('   ⚠️ No products found on this page!')
        return products

    # Loop through each price element and extract data
    for price_el in price_elements:

        # The product card is the grandparent of product-price
        # Structure: card > div.mb-4.grow > div.product-price
        card = price_el.parent.parent

        # -----------------------------------------------------------
        # GET PRODUCT NAME
        # -----------------------------------------------------------
        # Name is in <span class="line-clamp-2 text-ellipsis">
        # inside an <a> tag in the card
        # -----------------------------------------------------------
        name_el = card.select_one('.line-clamp-2')
        name = name_el.get_text(strip=True) if name_el else 'Not found'

        # -----------------------------------------------------------
        # GET CURRENT PRICE (the green price — what you pay)
        # -----------------------------------------------------------
        # Inside product-price, the current price is in:
        # <span class="font-bold text-naivas-green ...">
        # -----------------------------------------------------------
        current_price_el = price_el.select_one('.text-naivas-green')
        current_price = current_price_el.get_text(strip=True) if current_price_el else 'Not found'

        # -----------------------------------------------------------
        # GET ORIGINAL PRICE (the red crossed-out price)
        # Some products are not on sale so this may be empty
        # -----------------------------------------------------------
        original_price_el = price_el.select_one('.line-through')
        original_price = original_price_el.get_text(strip=True) if original_price_el else 'No discount'

        # -----------------------------------------------------------
        # GET PRODUCT LINK
        # -----------------------------------------------------------
        link_el = card.select_one('a')
        link = link_el.get('href', '') if link_el else ''

        # -----------------------------------------------------------
        # SAVE THIS PRODUCT
        # -----------------------------------------------------------
        product = {
            'Category'       : category_name,
            'Product Name'   : name,
            'Current Price'  : current_price,
            'Original Price' : original_price,
            'Product URL'    : link,
        }
        products.append(product)

    return products

print('✅ extract_products() function updated with correct selectors!')

✅ extract_products() function updated with correct selectors!


In [30]:
# =============================================================
# CELL 6 — RUN THE SCRAPER WITH PAGINATION
# =============================================================
# Previously we only scraped page 1 of each category.
# Now we loop through ALL pages until we find an empty page
# which tells us we have reached the end.
#
# How pagination works:
# Page 1 → https://naivas.online/food-cupboard
# Page 2 → https://naivas.online/food-cupboard?page=2
# Page 3 → https://naivas.online/food-cupboard?page=3
# ... and so on until a page returns 0 products
# =============================================================

all_products = []  # Master list — all products from all categories

# Maximum pages to scrape per category
# This is a safety limit so we don't accidentally loop forever
MAX_PAGES = 20

print('🚀 Starting the scraper with pagination...\n')
print('=' * 55)

for category_name, base_url in CATEGORIES.items():

    print(f'\n📦 Category: {category_name}')
    print('-' * 40)

    category_products = []  # Products for this category only
    page_number = 1         # Start from page 1

    # Keep looping through pages until we find an empty one
    while page_number <= MAX_PAGES:

        # ---------------------------------------------------
        # BUILD THE PAGE URL
        # ---------------------------------------------------
        # Page 1 uses the base URL (no ?page= needed)
        # Page 2+ adds ?page=NUMBER to the URL
        # ---------------------------------------------------
        if page_number == 1:
            url = base_url
        else:
            url = f'{base_url}?page={page_number}'

        print(f'\n   📄 Page {page_number}: {url}')

        # Fetch the page
        html = fetch_page(url)

        if not html:
            print(f'   ⚠️ Could not load page {page_number}. Stopping.')
            break  # Stop paginating this category

        # Extract products from this page
        products = extract_products(html, category_name)

        # ---------------------------------------------------
        # CHECK IF WE HAVE REACHED THE LAST PAGE
        # ---------------------------------------------------
        # If a page returns 0 products it means we have gone
        # past the last page — so we stop looping
        # ---------------------------------------------------
        if len(products) == 0:
            print(f'   🏁 No products on page {page_number}.')
            print(f'   🏁 Reached last page! Stopping pagination.')
            break

        # Add this page's products to our category list
        category_products.extend(products)
        print(f'   ✅ Page {page_number}: collected {len(products)} products')
        print(f'   📊 Running total for {category_name}: {len(category_products)}')

        # Move to the next page
        page_number += 1

        # Be polite — wait between page requests
        print(f'   ⏱️  Waiting {DELAY_SECONDS} seconds...')
        time.sleep(DELAY_SECONDS)

    # Add all this category's products to the master list
    all_products.extend(category_products)
    print(f'\n   ✅ Finished {category_name}: {len(category_products)} total products')

print('\n' + '=' * 55)
print(f'🎉 Scraping complete!')
print(f'   Total products collected: {len(all_products)}')

🚀 Starting the scraper with pagination...


📦 Category: Food Cupboard
----------------------------------------

   📄 Page 1: https://naivas.online/food-cupboard
   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
   🔍 Found 15 products on page
   ✅ Page 1: collected 15 products
   📊 Running total for Food Cupboard: 15
   ⏱️  Waiting 3 seconds...

   📄 Page 2: https://naivas.online/food-cupboard?page=2
   🌐 Visiting: https://naivas.online/food-cupboard?page=2
   📡 Response status: 200
   ✅ Page loaded successfully!
   🔍 Found 15 products on page
   ✅ Page 2: collected 15 products
   📊 Running total for Food Cupboard: 30
   ⏱️  Waiting 3 seconds...

   📄 Page 3: https://naivas.online/food-cupboard?page=3
   🌐 Visiting: https://naivas.online/food-cupboard?page=3
   📡 Response status: 200
   ✅ Page loaded successfully!
   🔍 Found 15 products on page
   ✅ Page 3: collected 15 products
   📊 Running total for Food Cupboard: 45
   ⏱️  Wai

In [31]:
# =============================================================
# CELL 7 — PREVIEW YOUR DATA
# =============================================================
# We turn our list into a pandas DataFrame — a table,
# just like a spreadsheet with rows and columns.
# =============================================================

# Create DataFrame (table) from our list of products
df = pd.DataFrame(all_products)

# .shape gives us (number of rows, number of columns)
print(f'📊 Table size: {df.shape[0]} products × {df.shape[1]} columns\n')

# Count how many products we got from each category
print('📋 Products per category:')
print(df['Category'].value_counts())
print()

# Show a summary of prices found vs not found
print('💰 Price summary:')
print(df['Current Price'].value_counts().head(5))
print()

# Show the first 10 rows to verify data looks correct
print('👀 Preview of first 10 products:')
df.head(10)

📊 Table size: 1156 products × 5 columns

📋 Products per category:
Category
Food Cupboard    300
Fresh Food       300
Naivas Liquor    300
Electronics      256
Name: count, dtype: int64

💰 Price summary:
Current Price
KES 179    13
KES 99     12
KES 199    11
KES 299    11
KES 159    10
Name: count, dtype: int64

👀 Preview of first 10 products:


,Category,Product Name,Current Price,Original Price,Product URL
0,Food Cupboard,Sunrice Basmati Rice 5Kg,"KES 1,299","KES 1,825",https://naivas.online/sunrice-basmati-rice-5kg
1,Food Cupboard,Lea Premium Maize Meal 2Kg,KES 193,No discount,https://naivas.online/lea-premium-maize-meal-2kg
2,Food Cupboard,Rina Vegetable Oil 5L,"KES 1,169","KES 1,600",https://naivas.online/rina-vegetable-oil-5l
3,Food Cupboard,Sunrice Biryani 5Kg,KES 999,"KES 1,150",https://naivas.online/sunrice-biryani-5kg
4,Food Cupboard,Blueband Peanut Butter 400ml,KES 290,KES 360,https://naivas.online/blueband-peanut-butter-4...
5,Food Cupboard,Tupike Maize Meal 2Kg,KES 139,KES 144,https://naivas.online/tupike-maize-meal-2kg
6,Food Cupboard,Dola Cooking Oil 5Ltr,"KES 1,099","KES 1,419",https://naivas.online/dola-cooking-oil-5ltr
7,Food Cupboard,Fresh Fri Vegetable Oil 5L,"KES 1,299","KES 1,600",https://naivas.online/fresh-fri-vegetable-oil-5l
8,Food Cupboard,Blueband Margarine 1Kg,KES 480,KES 585,https://naivas.online/blueband-margarine-1kg
9,Food Cupboard,Maryam Dates 905G,KES 745,KES 995,https://naivas.online/maryam-dates-905g


In [29]:
# =============================================================
# DIAGNOSTIC — Check pagination URL structure
# =============================================================
# We fetch page 2 of Food Cupboard to see if Naivas uses
# a standard ?page=2 pattern for multiple pages
# =============================================================

# Common pagination patterns to test
test_urls = [
    'https://naivas.online/food-cupboard?page=2',
    'https://naivas.online/food-cupboard?p=2',
]

for url in test_urls:
    html = fetch_page(url)
    if html:
        soup = BeautifulSoup(html, 'html.parser')
        prices = soup.select('.product-price')
        print(f'URL: {url}')
        print(f'Products found: {len(prices)}')
        print()
    time.sleep(3)

   🌐 Visiting: https://naivas.online/food-cupboard?page=2
   📡 Response status: 200
   ✅ Page loaded successfully!
URL: https://naivas.online/food-cupboard?page=2
Products found: 15

   🌐 Visiting: https://naivas.online/food-cupboard?p=2
   📡 Response status: 200
   ✅ Page loaded successfully!
URL: https://naivas.online/food-cupboard?p=2
Products found: 15



In [22]:
# =============================================================
# DIAGNOSTIC — Test extract_products on one page only
# =============================================================

html = fetch_page('https://naivas.online/food-cupboard')

if html:
    result = extract_products(html, 'Food Cupboard')
    print(f'Type returned: {type(result)}')
    print(f'Number of products: {len(result) if result else 0}')
    if result:
        print(f'First product: {result[0]}')

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
   🔍 Found 15 products on page
Type returned: <class 'NoneType'>
Number of products: 0


In [23]:
# Check the end of the extract_products function
import inspect
print(inspect.getsource(extract_products))

def extract_products(html, category_name):
    """
    Reads page HTML and extracts product names and prices.
    Returns a list of products (each product is a dictionary).
    """

    soup = BeautifulSoup(html, 'html.parser')
    products = []

    # -----------------------------------------------------------
    # FIND ALL PRODUCT CARDS
    # -----------------------------------------------------------
    # Each product lives inside a <div class="flex flex-col grow">
    # We find them by looking for the product-price div inside,
    # then navigating to its grandparent (the full card)
    # -----------------------------------------------------------

    price_elements = soup.select('.product-price')
    print(f'   🔍 Found {len(price_elements)} products on page')

    if not price_elements:
        print('   ⚠️ No products found on this page!')
        return products

    # Loop through each price element and extract data
    for price_el in price_elements:

        # The product 

In [8]:
# =============================================================
# DIAGNOSTIC CELL — Find the correct HTML selectors
# =============================================================
# This cell fetches one page and prints the HTML of the
# first product card so we can see the exact class names.
# DELETE or SKIP this cell after we fix the selectors.
# =============================================================

from bs4 import BeautifulSoup

# Fetch the food cupboard page
html = fetch_page('https://naivas.online/food-cupboard')
soup = BeautifulSoup(html, 'html.parser')

# Find all product cards
product_cards = soup.select('li.product')
print(f'Found {len(product_cards)} product cards\n')

# Print the raw HTML of just the FIRST product card
# This shows us exactly what class names are being used
if product_cards:
    print('=== FIRST PRODUCT CARD HTML ===\n')
    print(product_cards[0].prettify())  # prettify() makes it readable

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
Found 0 product cards



In [9]:
# =============================================================
# DIAGNOSTIC CELL 2 — Inspect the full page HTML
# =============================================================
# Since li.product returned nothing, we print a chunk of the
# actual page HTML so we can see what structure Naivas uses
# =============================================================

html = fetch_page('https://naivas.online/food-cupboard')
soup = BeautifulSoup(html, 'html.parser')

# Print the first 3000 characters of the page HTML
# This helps us see the structure near the product listings
print(soup.prettify()[:3000])

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
<!DOCTYPE html>
<html lang="en">
 <head>
  <title>
   Food Cupboard
  </title>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="en" http-equiv="content-language"/>
  <meta content="kmzypjIylropBFnROpDoiMmOtox750m2IViaX4cx" name="csrf-token"/>
  <meta content="https://naivas.online" name="base-url"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="" name="description">
   <meta content="" name="keywords">
    <script type="application/ld+json">
     {"@type":"WebSite","@context":"http:\/\/schema.org","url":"https:\/\/naivas.online","potentialAction":{"@type":"SearchAction","target":"https:\/\/naivas.online\/search\/?term={search_term_string}","query-input":"required name=search_term_string"}}
    </script>
    <link href="https://naivas.online/themes/default/assets/images/

In [10]:
# =============================================================
# DIAGNOSTIC CELL 3 — Search for product containers
# =============================================================
# Since this is not WooCommerce, we need to find what tags
# and class names Naivas actually uses for their products.
# This cell searches the HTML for anything product-related.
# =============================================================

html = fetch_page('https://naivas.online/food-cupboard')
soup = BeautifulSoup(html, 'html.parser')

# Print ALL unique class names found on the page
# This helps us spot which ones are product-related
print("=== ALL UNIQUE CLASS NAMES ON PAGE ===\n")

all_classes = set()
for tag in soup.find_all(True):  # find_all(True) = every single HTML tag
    for cls in tag.get('class', []):
        all_classes.add(cls)

# Sort and print them so they are easy to read
for cls in sorted(all_classes):
    print(cls)

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
=== ALL UNIQUE CLASS NAMES ON PAGE ===

!text-naivas-gray-dark
-ml-1
-ml-1.5
-mt-0.5
-mt-2
-mt-2.5
-px-px
-translate-x-full
2xl:gap-10
2xl:grid-cols-2
2xl:max-large-2xl:px-20
2xl:pt-[13rem]
__cf_email__
aa-ClearIcon
aa-Input
aa-SubmitIcon
absolute
align-items-center
align-middle
animate-spin
appearance-none
bg-[#F1F1F1]
bg-black/70
bg-naivas-background
bg-naivas-bg
bg-naivas-gray-light
bg-naivas-green
bg-naivas-orange
bg-naivas-orange/10
bg-transparent
bg-white
block
bold
border
border-2
border-4
border-[#069680]
border-[0.75px]
border-b
border-b-0
border-b-2
border-b-white
border-gray-100
border-gray-200
border-gray-300
border-l-0
border-naivas-bg
border-naivas-border
border-naivas-color
border-naivas-gray-dark/15
border-naivas-gray-light
border-naivas-green
border-naivas-orange
border-none
border-r-0
border-t
border-t-1.5
bottom-0
btn-age-gate
caret-[#069680]
category-descripti

In [11]:
# =============================================================
# DIAGNOSTIC CELL 4 — Test the correct selectors
# =============================================================
# We spotted 'product-price' and 'item' in the class list.
# Let's test them and print what they contain.
# =============================================================

html = fetch_page('https://naivas.online/food-cupboard')
soup = BeautifulSoup(html, 'html.parser')

# Test 1: Find product cards using 'item' class
cards = soup.select('.item')
print(f"Cards found with '.item': {len(cards)}")

# Print first card's HTML to see its structure
if cards:
    print("\n=== FIRST CARD HTML ===")
    print(cards[0].prettify())
    print("\n=== SECOND CARD HTML ===")
    print(cards[1].prettify())

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
Cards found with '.item': 114

=== FIRST CARD HTML ===
<li class="mb-3 item">
 <div class="font-light text-gray-500 hover:text-naivas-green">
  Food cupboard deals
 </div>
</li>


=== SECOND CARD HTML ===
<li class="mb-3 item">
 <div class="font-light text-gray-500 hover:text-naivas-green">
  Fresh Deals
 </div>
</li>



In [12]:
# =============================================================
# DIAGNOSTIC CELL 5 — Find the actual product cards
# =============================================================
# .item found menu items not products. Let's look for
# product-price class since we know that one exists,
# then go UP to find its parent container (the product card)
# =============================================================

html = fetch_page('https://naivas.online/food-cupboard')
soup = BeautifulSoup(html, 'html.parser')

# Find elements with 'product-price' class
prices = soup.select('.product-price')
print(f"Price elements found: {len(prices)}")

if prices:
    print("\n=== FIRST PRICE ELEMENT ===")
    print(prices[0].prettify())
    
    print("\n=== PARENT OF FIRST PRICE (the product card) ===")
    print(prices[0].parent.prettify())
    
    print("\n=== GRANDPARENT (one level higher) ===")
    print(prices[0].parent.parent.prettify())

   🌐 Visiting: https://naivas.online/food-cupboard
   📡 Response status: 200
   ✅ Page loaded successfully!
Price elements found: 15

=== FIRST PRICE ELEMENT ===
<div class="product-price">
 <p class="my-0 leading-none flex flex-col md:flex-row md:items-center">
  <span class="font-bold text-naivas-green mb-1 md:mb-0 pe-2">
   KES 999
  </span>
  <span class="text-red-600 text-xs line-through font-light">
   KES 1,825
  </span>
 </p>
 <p class="leading-normal">
  <span class="text-xxs text-black-50 font-medium">
   Save KES 826
  </span>
 </p>
</div>


=== PARENT OF FIRST PRICE (the product card) ===
<div class="mb-4 grow">
 <div class="product-price">
  <p class="my-0 leading-none flex flex-col md:flex-row md:items-center">
   <span class="font-bold text-naivas-green mb-1 md:mb-0 pe-2">
    KES 999
   </span>
   <span class="text-red-600 text-xs line-through font-light">
    KES 1,825
   </span>
  </p>
  <p class="leading-normal">
   <span class="text-xxs text-black-50 font-medium">
 

In [33]:
# =============================================================
# CELL 8 — SAVE YOUR DATA TO FILES
# =============================================================
# We save the data in two formats:
# 1. CSV  — a simple text file, opens in Excel or Notepad
# 2. Excel — a proper .xlsx file for Excel/Google Sheets
#
# Files are saved in the same folder as this notebook.
# =============================================================

# --- SAVE AS CSV ---
# encoding='utf-8-sig' makes sure special characters (like
# currency symbols KES) display correctly when opened in Excel.
# index=False means don't add an extra row-number column.
csv_file = 'naivas_products.csv'
df.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f'💾 Saved CSV  → {csv_file}')

# --- SAVE AS EXCEL ---
excel_file = 'naivas_products.xlsx'
df.to_excel(excel_file, index=False)
print(f'💾 Saved Excel → {excel_file}')

print('\n✅ All done! Your files are saved in your Jupyter folder.')
print('   Open naivas_products.xlsx in Microsoft Excel to view your data!')

💾 Saved CSV  → naivas_products.csv
💾 Saved Excel → naivas_products.xlsx

✅ All done! Your files are saved in your Jupyter folder.
   Open naivas_products.xlsx in Microsoft Excel to view your data!
